<a href="https://colab.research.google.com/github/raheelam98/DataAnalysis/blob/main/EDA/EDA_Sessions_RM/Data_Cleaning_Inconsistent_Data_Sec_4_RM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, we're going to learn how to clean up inconsistent text entries.

Let's get started!

# Get our environment set up

The first thing we'll need to do is load in the libraries and dataset we'll be using.

In [ ]:
#isl
#islamabad
#ISL
#ISLAMABAD
#islamaBad
#5 unique---- 1 unique
a="Isl"
a.replace("isl","islamabad")
print(a)
b="Islamabad"
b.lower()

print(b)
#finds similar strings of a category
#islamabad- isl, ISLAMABAD--- matching

Isl
Islamabad


In [ ]:
!pip install fuzzywuzzy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# modules we'll use
import pandas as pd
import numpy as np

# helpful modules
import fuzzywuzzy # pip install fuzzywuzzy
from fuzzywuzzy import process

# read in all our data
professors = pd.read_csv('/content/drive/MyDrive/EDA DA5/pakistan_intellectual_capital.csv')
professors

,Unnamed: 0,S#,Teacher Name,University Currently Teaching,Department,Province University Located,Designation,Terminal Degree,Graduated from,Country,Year,Area of Specialization/Research Interests,Other Information
0,2,3,Dr. Abdul Basit,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,Software Engineering & DBMS,NaN
1,4,5,Dr. Waheed Noor,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,DBMS,NaN
2,5,6,Dr. Junaid Baber,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"Information processing, Multimedia mining",NaN
3,6,7,Dr. Maheen Bakhtyar,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"NLP, Information Retrieval, Question Answering...",NaN
4,24,25,Samina Azim,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Balochistan University of Information Technolo...,Pakistan,2005.0,VLSI Electronics DLD Database,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1137,1974,1975,Dr. Ahmar Rashid,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Associate Professor,PhD,JNU,South Korea,NaN,"Electrical Impedance Tomography, Inverse algor...",NaN
1138,1975,1976,Dr. Fawad Hussain,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Associate Professor,PhD,Grenoble,France,NaN,"Machine Learning, Big Data Anaysis, Data Minin...",NaN
1139,1977,1978,Dr. Rashad M Jillani,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Assistant Professor,PhD,Florida Atlantic University,USA,2012.0,"Digital Multimedia Systems, Video Compression ...",NaN
1140,1979,1980,Dr. Shahabuddin Ansari,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Assistant Professor,PhD,Ghulam Ishaq Khan Institute of Science and Tec...,Pakistan,NaN,"Medical Image Processing and Analysis, Digital...",NaN


In [ ]:
professors.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1142 entries, 0 to 1141
Data columns (total 13 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Unnamed: 0                                 1142 non-null   int64  
 1   S#                                         1142 non-null   int64  
 2   Teacher Name                               1142 non-null   object 
 3   University Currently Teaching              1142 non-null   object 
 4   Department                                 1142 non-null   object 
 5   Province University Located                1142 non-null   object 
 6   Designation                                1123 non-null   object 
 7   Terminal Degree                            1138 non-null   object 
 8   Graduated from                             1142 non-null   object 
 9   Country                                    1142 non-null   object 
 10  Year                    

In [ ]:
professors.isnull().sum()
#1142

,0
Unnamed: 0,0
S#,0
Teacher Name,0
University Currently Teaching,0
Department,0
Province University Located,0
Designation,19
Terminal Degree,4
Graduated from,0
Country,0


In [ ]:
#finding missing values in all columns
#area - remove-- formating type
#year- replace with mode? "replace it" unknown
#other info- remove

In [ ]:
#,index_col="Teacher Name"
professors.columns

Index(['Unnamed: 0', 'S#', 'Teacher Name', 'University Currently Teaching',
       'Department', 'Province University Located', 'Designation',
       'Terminal Degree', 'Graduated from', 'Country', 'Year',
       'Area of Specialization/Research Interests', 'Other Information'],
      dtype='object')

In [ ]:
#1142 entries--- column-- 1142 unique values

In [ ]:
professors['Teacher Name'].nunique()

1133

In [ ]:
professors['S#'].nunique()

1142

In [ ]:
professors['Unnamed: 0'].nunique()

1142

In [ ]:
professors['S#'].duplicated()

,S#
0,False
1,False
2,False
3,False
4,False
...,...
1137,False
1138,False
1139,False
1140,False


In [ ]:
professors['S#'].duplicated().sum()

np.int64(0)

# Do some preliminary text pre-processing

We'll begin by taking a quick look at the first few rows of the data.

In [ ]:
professors.head(20)

,Unnamed: 0,S#,Teacher Name,University Currently Teaching,Department,Province University Located,Designation,Terminal Degree,Graduated from,Country,Year,Area of Specialization/Research Interests,Other Information
0,2,3,Dr. Abdul Basit,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,Software Engineering & DBMS,NaN
1,4,5,Dr. Waheed Noor,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,DBMS,NaN
2,5,6,Dr. Junaid Baber,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"Information processing, Multimedia mining",NaN
3,6,7,Dr. Maheen Bakhtyar,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"NLP, Information Retrieval, Question Answering...",NaN
4,24,25,Samina Azim,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Balochistan University of Information Technolo...,Pakistan,2005.0,VLSI Electronics DLD Database,NaN
5,25,26,Nausheed Saeed,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,MCS,University of Balochistan,Pakistan,2008.0,"Software Engineering, Computer Networks.",NaN
6,26,27,Shumaila Hussain,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,MS,Balochistan University of Information Technolo...,Pakistan,2011.0,"Human computer Interaction, Web Development, S...",NaN
7,27,28,Mirfa Manzoor,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Sardar Bahadur Khan Women's University,Pakistan,2009.0,"Human Computer Interaction, Web",NaN
8,28,29,Saira Mujahid,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Sardar Bahadur Khan Women's University,Pakistan,2009.0,JAVA Programming,NaN
9,29,30,Arifa Anwar,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Sardar Bahadur Khan Women's University,Pakistan,2009.0,Programming,NaN


In [ ]:
professors.drop(['S#', 'Unnamed: 0','Area of Specialization/Research Interests'	,'Other Information',"Teacher Name"], axis=1,inplace=True)
professors

,University Currently Teaching,Department,Province University Located,Designation,Terminal Degree,Graduated from,Country,Year
0,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN
1,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN
2,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN
3,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN
4,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Balochistan University of Information Technolo...,Pakistan,2005.0
...,...,...,...,...,...,...,...,...
1137,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Associate Professor,PhD,JNU,South Korea,NaN
1138,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Associate Professor,PhD,Grenoble,France,NaN
1139,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Assistant Professor,PhD,Florida Atlantic University,USA,2012.0
1140,Ghulam Ishaq Khan Institute,Computer Science and Engineering,KPK,Assistant Professor,PhD,Ghulam Ishaq Khan Institute of Science and Tec...,Pakistan,NaN


In [ ]:
professors.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1142 entries, 0 to 1141
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Teacher Name                   1142 non-null   object 
 1   University Currently Teaching  1142 non-null   object 
 2   Department                     1142 non-null   object 
 3   Province University Located    1142 non-null   object 
 4   Designation                    1123 non-null   object 
 5   Terminal Degree                1138 non-null   object 
 6   Graduated from                 1142 non-null   object 
 7   Country                        1142 non-null   object 
 8   Year                           489 non-null    float64
dtypes: float64(1), object(8)
memory usage: 80.4+ KB


In [ ]:
professors["Terminal Degree"].value_counts()

Terminal Degree
MS                                463
PhD                               363
BS                                 93
MSc                                37
MCS                                20
Mphil                              17
M.E                                14
MSCS                               13
Post Doc                           13
M.Phil                             13
Phd                                12
MBA                                11
ME                                 11
BE                                  6
PostDoc                             6
MPhil                               4
BCS                                 4
MSSE                                3
MA                                  3
Bachelors                           3
M.Sc                                3
MCIT                                3
MSC                                 2
Ph.D                                2
MSIT                                2
MIT                               

In [ ]:
for column in professors:
    print("Column"," ",column,"  uniquevalue  are :", professors[column].nunique())
    print(professors[column].unique())
    print("-------------------------")

Column   University Currently Teaching   uniquevalue  are : 63
['University of Balochistan' "Sardar Bahadur Khan Women's University"
 'University of Turbat' 'COMSATS, Islamabad Campus'
 'National University of Sciences and Technology' 'RIPHAH International'
 'FAST-NU(Islamabad)' 'International Islamic University,Islamabad'
 'National University of Modern Languages' 'Air University'
 'Bahria University,Islamabad'
 'Capital University of Science and Technology'
 'Pakistan Institute of Engineering and Applied Sciences'
 'University of Sargodha,Mandi Bahauddin Campus' 'NAML-Mianwali'
 'University of Lahore-PakPattan' 'University of Sahiwal'
 'Barani Institute of Information and Technology'
 'Fatima Jinnah Women University' 'FAST(Faisalabad)'
 'University of Central Punjab' 'Lahore Garrison University'
 'Punjab University College of Information and Technology' 'FAST(Lahore)'
 'Information Technology University'
 'Lahore University of Management Sciences' 'Virtual University'
 'University of

Say we're interested in cleaning up the "Country" column to make sure there's no data entry inconsistencies in it. We could go through and check each row by hand, of course, and hand-correct inconsistencies when we find them. There's a more efficient way to do this, though!

In [ ]:
professors['Department'].unique()

array(['Computer Science & IT', 'Computer Science', 'Computing',
       'Computer Science and Software Engineering',
       'Computer and Information Sciences', 'Computer Science and IT',
       'Software Engineering', 'Computer Sciences',
       'Information Technology', 'CS & IT',
       'School of Information and Technology',
       'Institute of Mathematics & Computer Science',
       'ENGINEERING, SCIENCE & TECHNOLOGY', 'Computer Engineering',
       'Computing & Information Sciences',
       'FACULTY OF COMPUTING & ENGINEERING',
       'Computer Science and Engineering'], dtype=object)

In [ ]:
# get all the unique values in the 'Country' column
professors['Department']=professors['Department'].str.lower()
departments = professors['Department'].unique()

# sort them alphabetically and then take a closer look
departments.sort()
departments

array(['computer and information sciences', 'computer engineering',
       'computer science', 'computer science & it',
       'computer science and engineering', 'computer science and it',
       'computer science and software engineering', 'computer sciences',
       'computing', 'computing & information sciences', 'cs & it',
       'engineering, science & technology',
       'faculty of computing & engineering', 'information technology',
       'institute of mathematics & computer science',
       'school of information and technology', 'software engineering'],
      dtype=object)

In [ ]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries.sort()
countries

array(['Australia', 'Austria', 'Canada', 'China', 'Finland', 'France',
       'Germany', 'Greece', 'HongKong', 'Ireland', 'Italy', 'Japan',
       'Macau', 'Malaysia', 'Mauritius', 'Netherland', 'New Zealand',
       'Norway', 'Pakistan', 'Portugal', 'Russian Federation',
       'Saudi Arabia', 'Scotland', 'Singapore', 'South Korea',
       'SouthKorea', 'Spain', 'Sweden', 'Thailand', 'Turkey', 'UK', 'USA',
       'USofA', 'Urbana', 'germany'], dtype=object)

In [ ]:
countries = professors['Country'].str.lower()

# sort them alphabetically and then take a closer look

countries.unique()

array(['thailand', 'pakistan', 'germany', 'austria', 'australia', 'uk',
       'china', 'france', 'usofa', 'southkorea', 'malaysia', 'sweden',
       'italy', 'canada', 'norway', 'ireland', 'new zealand', 'urbana',
       'portugal', 'russian federation', 'usa', 'finland', 'netherland',
       'greece', 'turkey', 'south korea', 'macau', 'singapore', 'spain',
       'japan', 'hongkong', 'saudi arabia', 'mauritius', 'scotland'],
      dtype=object)

In [ ]:
# get the top 10 closest matches to "CS"
matchesdep = fuzzywuzzy.process.extract("computer sciences", departments, limit=20)

# take a look at them
matchesdep


[('computer sciences', 100),
 ('computer science', 97),
 ('institute of mathematics & computer science', 87),
 ('computer and information sciences', 86),
 ('computer science and engineering', 86),
 ('computer science and software engineering', 86),
 ('computing & information sciences', 86),
 ('computer science & it', 85),
 ('computer science and it', 80),
 ('computer engineering', 65),
 ('computing', 62),
 ('engineering, science & technology', 53),
 ('faculty of computing & engineering', 50),
 ('cs & it', 34),
 ('software engineering', 32),
 ('information technology', 31),
 ('school of information and technology', 30)]

In [ ]:
# get the top 10 closest matches to "south korea"
matchesdep = fuzzywuzzy.process.extract("computer", departments, limit=15)

# take a look at them
matchesdep
#percentage 68%


[('computer and information sciences', 90),
 ('computer engineering', 90),
 ('computer science', 90),
 ('computer science & it', 90),
 ('computer science and engineering', 90),
 ('computer science and it', 90),
 ('computer science and software engineering', 90),
 ('computer sciences', 90),
 ('institute of mathematics & computer science', 90),
 ('computing', 71),
 ('computing & information sciences', 68),
 ('faculty of computing & engineering', 68),
 ('school of information and technology', 43),
 ('engineering, science & technology', 34),
 ('information technology', 34)]

In [ ]:
#Terminal Degree
# get the top 10 closest matches to "south korea"
matchesdep = fuzzywuzzy.process.extract('BE',Terminal Degree,limit=15)

# take a look at them
matchesdep
#percentage 68%

SyntaxError: invalid syntax. Perhaps you forgot a comma? (ipython-input-51-3404763809.py, line 3)

In [ ]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries.sort()
countries

array(['Australia', 'Austria', 'Canada', 'China', 'Finland', 'France',
       'Germany', 'Greece', 'HongKong', 'Ireland', 'Italy', 'Japan',
       'Macau', 'Malaysia', 'Mauritius', 'Netherland', 'New Zealand',
       'Norway', 'Pakistan', 'Portugal', 'Russian Federation',
       'Saudi Arabia', 'Scotland', 'Singapore', 'South Korea',
       'SouthKorea', 'Spain', 'Sweden', 'Thailand', 'Turkey', 'UK', 'USA',
       'USofA', 'Urbana', 'germany'], dtype=object)

Just looking at this, I can see some problems due to inconsistent data entry: ' Germany', and 'germany', for example, or ' New Zealand' and 'New Zealand'.

The first thing I'm going to do is make everything lower case (I can change it back at the end if I like) and remove any white spaces at the beginning and end of cells. Inconsistencies in capitalizations and trailing white spaces are very common in text data and you can fix a good 80% of your text data entry inconsistencies by doing this.

In [ ]:
# convert to lower case
professors['Country'] = professors['Country'].str.lower()
# remove trailing white spaces
professors['Country'] = professors['Country'].str.strip()


In [ ]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries.sort()
countries

array(['australia', 'austria', 'canada', 'china', 'finland', 'france',
       'germany', 'greece', 'hongkong', 'ireland', 'italy', 'japan',
       'macau', 'malaysia', 'mauritius', 'netherland', 'new zealand',
       'norway', 'pakistan', 'portugal', 'russian federation',
       'saudi arabia', 'scotland', 'singapore', 'south korea',
       'southkorea', 'spain', 'sweden', 'thailand', 'turkey', 'uk',
       'urbana', 'usa', 'usofa'], dtype=object)

Next we're going to tackle more difficult inconsistencies.

# Use fuzzy matching to correct inconsistent data entry

Alright, let's take another look at the 'Country' column and see if there's any more data cleaning we need to do.

In [ ]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries.sort()
countries

array(['australia', 'austria', 'canada', 'china', 'finland', 'france',
       'germany', 'greece', 'hongkong', 'ireland', 'italy', 'japan',
       'macau', 'malaysia', 'mauritius', 'netherland', 'new zealand',
       'norway', 'pakistan', 'portugal', 'russian federation',
       'saudi arabia', 'scotland', 'singapore', 'south korea',
       'southkorea', 'spain', 'sweden', 'thailand', 'turkey', 'uk',
       'urbana', 'usa', 'usofa'], dtype=object)

It does look like there is another inconsistency: 'southkorea' and 'south korea' should be the same.

We're going to use the [fuzzywuzzy](https://github.com/seatgeek/fuzzywuzzy) package to help identify which strings are closest to each other. This dataset is small enough that we could probably could correct errors by hand, but that approach doesn't scale well. (Would you want to correct a thousand errors by hand? What about ten thousand? Automating things as early as possible is generally a good idea. Plus, it’s fun!)

> **Fuzzy matching:** The process of automatically finding text strings that are very similar to the target string. In general, a string is considered "closer" to another one the fewer characters you'd need to change if you were transforming one string into another. So "apple" and "snapple" are two changes away from each other (add "s" and "n") while "in" and "on" and one change away (rplace "i" with "o"). You won't always be able to rely on fuzzy matching 100%, but it will usually end up saving you at least a little time.

Fuzzywuzzy returns a ratio given two strings. The closer the ratio is to 100, the smaller the edit distance between the two strings. Here, we're going to get the ten strings from our list of cities that have the closest distance to "south korea".

In [ ]:
# get the top 10 closest matches to "south korea"
matches = fuzzywuzzy.process.extract("usa", countries, limit=4)

# take a look at them
matches

[('usa', 100, 351), ('usa', 100, 377), ('usa', 100, 405), ('usa', 100, 409)]

In [ ]:
# get the top 10 closest matches to "south korea"
matches = fuzzywuzzy.process.extract("south korea", countries, limit=4)

# take a look at them
matches

[('south korea', 100, 489),
 ('south korea', 100, 677),
 ('south korea', 100, 685),
 ('south korea', 100, 764)]

We can see that two of the items in the cities are very close to "south korea": "south korea" and "southkorea". Let's replace all rows in our "Country" column that have a match of > 95 with "south korea".

To do this, I'm going to write a function. (It's a good idea to write a general purpose function you can reuse if you think you might have to do a specific task more than once or twice. This keeps you from having to copy and paste code too often, which saves time and can help prevent mistakes.)

In [ ]:
# function to replace rows in the provided column of the provided dataframe
# that match the provided string above the provided ratio with the provided string
def replace_matches_in_column(df, column, string_to_match, min_match = 95):
    # get a list of unique strings
    strings = df[column].unique()

    # get the top 10 closest matches to our input string
    matches = fuzzywuzzy.process.extract(string_to_match, strings,
                                         limit=5)

    # only get matches with a ratio > 95
    close_matches = [matches[0] for matches in matches if matches[1] >= min_match]

    # get the rows of all the close matches in our dataframe
    rows_with_matches = df[column].isin(close_matches)

    # replace all rows with close matches with the input matches
    df.loc[rows_with_matches, column] = string_to_match

    # let us know the function's done
    print("All done!")

Function Definition:

replace_matches_in_column(df, column, string_to_match, min_match=95): This function takes four parameters:
df: The dataframe where replacements will occur.
column: The column within the dataframe where you want to check for matches.
string_to_match: The string you want to find matches for.
min_match: The minimum similarity percentage to consider a string a match (default is 95%).
Extract Unique Strings:

strings = df[column].unique(): Retrieves all unique values from the specified column of the dataframe.
Find Close Matches:

matches = fuzzywuzzy.process.extract(string_to_match, strings, limit=5): Uses fuzzy string matching to find the top 5 strings from strings that are most similar to string_to_match.
Filter Matches by Similarity Threshold:

close_matches = [matches[0] for matches in matches if matches[1] >= min_match]: Filters out the matches to include only those with a similarity score greater than or equal to min_match. It creates a list of just these matching strings.
Identify Rows with Close Matches:

rows_with_matches = df[column].isin(close_matches): Creates a boolean series that is True for rows where the value in the specified column is in the close_matches list.
Replace Matched Rows:

df.loc[rows_with_matches, column] = string_to_match: Replaces the values in the matched rows’ specified column with string_to_match.
Completion Message:

print("All done!"): Prints a message indicating that the function has completed its execution.

In [ ]:
replace_matches_in_column(df=professors, column='Department', string_to_match="computer sciences",min_match = 97)
professors.Department.unique()

All done!


array(['computer science & it', 'computer sciences', 'computing',
       'computer science and software engineering',
       'computer and information sciences', 'computer science and it',
       'software engineering', 'information technology', 'cs & it',
       'school of information and technology',
       'institute of mathematics & computer science',
       'engineering, science & technology', 'computer engineering',
       'computing & information sciences',
       'faculty of computing & engineering',
       'computer science and engineering'], dtype=object)

Now that we have a function, we can put it to the test!

In [ ]:
# use the function we just wrote to replace close matches to "south korea" with "south korea"
replace_matches_in_column(df=professors, column='Country', string_to_match="south korea",min_match = 95)
professors.Country.unique()

All done!


array(['Thailand', 'Pakistan', 'germany', 'Austria', 'Australia', 'UK',
       'China', 'France', 'usa', 'south korea', 'Malaysia', 'Sweden',
       'Italy', 'Canada', 'Norway', 'Ireland', 'New Zealand', 'Urbana',
       'Portugal', 'Russian Federation', 'Finland', 'Netherland',
       'Germany', 'Greece', 'Turkey', 'Macau', 'Singapore', 'Spain',
       'Japan', 'HongKong', 'Saudi Arabia', 'Mauritius', 'Scotland'],
      dtype=object)

In [ ]:
# use the function we just wrote to replace close matches to "south korea" with "south korea"
replace_matches_in_column(df=professors, column='Country', string_to_match="usa",min_match = 75)
professors.Country.unique()

All done!


array(['Thailand', 'Pakistan', 'germany', 'Austria', 'Australia', 'UK',
       'China', 'France', 'usa', 'south korea', 'Malaysia', 'Sweden',
       'Italy', 'Canada', 'Norway', 'Ireland', 'New Zealand', 'Urbana',
       'Portugal', 'Russian Federation', 'Finland', 'Netherland',
       'Germany', 'Greece', 'Turkey', 'Macau', 'Singapore', 'Spain',
       'Japan', 'HongKong', 'Saudi Arabia', 'Mauritius', 'Scotland'],
      dtype=object)

And now let's check the unique values in our "Country" column again and make sure we've tidied up "south korea" correctly.

In [ ]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries.sort()
countries

array(['australia', 'austria', 'canada', 'china', 'finland', 'france',
       'germany', 'greece', 'hongkong', 'ireland', 'italy', 'japan',
       'macau', 'malaysia', 'mauritius', 'netherland', 'new zealand',
       'norway', 'pakistan', 'portugal', 'russian federation',
       'saudi arabia', 'scotland', 'singapore', 'south korea', 'spain',
       'sweden', 'thailand', 'turkey', 'uk', 'urbana', 'usa'],
      dtype=object)